In [1]:
import time
import pandas as pd
import requests
from tqdm import tqdm
from typing import Optional
# -----------------------------
# 1) KRX 종목 리스트
# -----------------------------
def read_krx_code():
    """
    KRX로부터 상장기업 목록을 읽어와 데이터프레임으로 반환
    """
    url = "http://kind.krx.co.kr/corpgeneral/corpList.do?method=download&searchType=13"
    krx = pd.read_html(url, header=0, encoding="euc-kr")[0]
    krx = krx[["종목코드", "회사명"]].rename(columns={"종목코드": "code", "회사명": "company"})
    krx["code"] = krx["code"].astype(str).str.zfill(6)
    return krx


# -----------------------------
# 2) 네이버 일별시세(HTML) 기반 OHLCV 수집
# -----------------------------
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Referer": "https://finance.naver.com/"
}

def fetch_naver_daily_ohlcv(code: str, pages: int = 20, sleep_sec: float = 0.2,
                           session: Optional[requests.Session] = None) -> pd.DataFrame:
    """
    네이버 금융 '일별시세' HTML 페이지에서 날짜별 OHLCV를 가져옵니다.
    (Python 3.10+ 타입힌트 문법이므로, Python 3.9면 아래 Optional로 바꿔주세요)
    """
    code = str(code).zfill(6)
    base_url = "https://finance.naver.com/item/sise_day.nhn"

    sess = session or requests.Session()
    all_parts = []

    for page in range(1, pages + 1):
        r = sess.get(base_url, params={"code": code, "page": page}, headers=HEADERS, timeout=20)
        r.raise_for_status()

        tables = pd.read_html(r.text, encoding="euc-kr")
        if not tables:
            break

        df = tables[0].copy()
        df = df.dropna(subset=["날짜"])
        if df.empty:
            break

        df = df.rename(columns={
            "날짜": "date",
            "시가": "open",
            "고가": "high",
            "저가": "low",
            "종가": "close",
            "거래량": "volume",
        })

        df = df[["date", "open", "high", "low", "close", "volume"]]
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        for c in ["open", "high", "low", "close", "volume"]:
            df[c] = pd.to_numeric(df[c], errors="coerce")

        df = df.dropna(subset=["date"])
        all_parts.append(df)

        time.sleep(sleep_sec)

    if not all_parts:
        return pd.DataFrame(columns=["date", "open", "high", "low", "close", "volume"])

    out = pd.concat(all_parts, ignore_index=True)
    out = out.drop_duplicates(subset=["date"]).sort_values("date").reset_index(drop=True)
    return out


# -----------------------------
# 3) 여러 code를 돌면서 결합
# -----------------------------
def fetch_naver_daily_ohlcv_all(
    pages: int = 20,
    limit: Optional[int] = None,     # 테스트용: 앞에서 N개만
    sleep_per_code: float = 0.3,     # 종목 간 추가 딜레이
    sleep_per_page: float = 0.2,     # 페이지 요청 간 딜레이
    verbose_fail: bool = True
) -> pd.DataFrame:
    """
    KRX 전체(또는 일부) 종목에 대해 네이버 일봉 OHLCV를 수집 후 하나의 DF로 결합.
    반환 컬럼: code, company, date, open, high, low, close, volume
    """
    krx = read_krx_code()
    krx = krx[~krx["company"].str.contains("스펙", na=False)].reset_index(drop=True)
    if limit is not None:
        krx = krx.head(limit)

    results = []
    with requests.Session() as sess:
        for row in tqdm(krx.itertuples(index=False), total=len(krx), desc="Collecting Naver OHLCV"):
            code = row.code
            company = row.company

            try:
                df = fetch_naver_daily_ohlcv(
                    code=code,
                    pages=pages,
                    sleep_sec=sleep_per_page,
                    session=sess
                )

                if not df.empty:
                    df.insert(0, "company", company)
                    df.insert(0, "code", code)
                    results.append(df)

            except Exception as e:
                if verbose_fail:
                    print(f"[WARN] code={code} ({company}) failed: {e}")

            time.sleep(sleep_per_code)

    if not results:
        return pd.DataFrame(columns=["code", "company", "date", "open", "high", "low", "close", "volume"])

    out = pd.concat(results, ignore_index=True)
    out = out.sort_values(["code", "date"]).reset_index(drop=True)
    return out

In [3]:
df_all = fetch_naver_daily_ohlcv_all(
    pages=6,
    sleep_per_page=0.3,
    sleep_per_code=0.5
)

KeyboardInterrupt: 

In [18]:
krx = read_krx_code()
krx

C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\bs4\__init__.py:339: UserWarning: You provided Unicode markup but also provided a value for from_encoding. Your from_encoding will be ignored.
  warnings.warn(


,code,company
0,012210,삼미금속
1,490470,세미파이브
2,491000,리브스메드
3,0097F0,미래에셋비전스팩10호
4,0099W0,미래에셋비전스팩11호
...,...,...
2783,000100,유한양행
2784,000120,CJ대한통운
2785,000050,경방
2786,000700,유수홀딩스
